# CLUSTER BACKTEST — does the GEHC filter work historically? (mechanical version)\nTests: >=3 distinct discretionary insider buyers, >=$500K, non-10b5-1, inside a 25%+ drawdown episode, entry at next close within 25% of insiders' avg price → forward 6/12m vs SPY.\n\n**PRE-REGISTERED (2026-07-12, before first run):** H1 positive 12m alpha vs SPY, low-to-mid single digits median, ~55-60% win rate. H2 bigger clusters / more buyers → stronger. H3 a meaningful minority (~15-25%) are FISV-type traps (fwd12m < -25%) — the judgment layer's job. \n\n**Limits stated:** survivorship (current index members); tests the MECHANICAL funnel only (no E-path judgment); health gates omitted in v1 (add via companyfacts if v1 shows promise). **Runtime: Cell 2 is HOURS for sp500/10y** (EDGAR rate limits) — run it and walk away; it checkpoints, so re-running the cell resumes rather than restarts.

In [ ]:
# 1 · UNIVERSE + DRAWDOWN EPISODES (the cheap gate first — only parse insiders inside episodes)
%pip -q install yfinance
import pandas as pd, numpy as np, yfinance as yf, requests
from io import StringIO

UNIVERSE = ["sp500"]                 # add "sp400" for a bigger (slower) run
START = "2015-01-01"                 # backtest window start
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
WIKI = {"sp500": "List_of_S%26P_500_companies", "sp400": "List_of_S%26P_400_companies"}
tickers = []
for u in UNIVERSE:
    html = requests.get(f"https://en.wikipedia.org/wiki/{WIKI[u]}", headers=UA, timeout=30).text
    tickers += [str(t).replace(".", "-").strip() for t in pd.read_html(StringIO(html))[0]["Symbol"]]
tickers = sorted(set(tickers))
px = yf.download(tickers, start=START, auto_adjust=True, progress=True)["Close"].dropna(axis=1, how="all")
spy = yf.download("SPY", start=START, auto_adjust=True, progress=False)["Close"].squeeze()
print(f"{px.shape[1]} names priced")

# drawdown episodes: >=25% below rolling 2y high; episode = contiguous run (merge gaps < 21 trading days)
episodes = []
for t in px.columns:
    s = px[t].dropna()
    if len(s) < 300: continue
    dd = s / s.rolling(504, min_periods=126).max() - 1
    in_dd = dd <= -0.25
    if not in_dd.any(): continue
    grp = (in_dd != in_dd.shift()).cumsum()
    for _, seg in s[in_dd].groupby(grp[in_dd]):
        if len(seg) < 10: continue
        episodes.append({"ticker": t, "ep_start": seg.index[0], "ep_end": seg.index[-1]})
ep = pd.DataFrame(episodes)
# merge episodes separated by <21 trading days
merged = []
for t, g in ep.groupby("ticker"):
    g = g.sort_values("ep_start")
    cur = None
    for _, r in g.iterrows():
        if cur is None: cur = r.to_dict(); continue
        if (r.ep_start - cur["ep_end"]).days <= 30:
            cur["ep_end"] = r.ep_end
        else:
            merged.append(cur); cur = r.to_dict()
    if cur is not None: merged.append(cur)
ep = pd.DataFrame(merged)
print(f"{len(ep)} drawdown episodes across {ep.ticker.nunique()} tickers")


In [ ]:
# 2 · CLUSTER DETECTION INSIDE EPISODES — EDGAR Form 4 parse (the slow cell: hours for sp500/10y)
# Checkpointed: reruns skip already-parsed tickers. Entry = close on day AFTER the qualifying filing.
import requests, time, re
import xml.etree.ElementTree as ET

H = {"User-Agent": "Jake research 7bm4q6x5sm@privaterelay.appleid.com"}
cm = requests.get("https://www.sec.gov/files/company_tickers.json", headers=H, timeout=30).json()
cikmap = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in cm.values()}

def txt(el, path):
    e = el.find(path)
    return e.text.strip() if e is not None and e.text else None

def all_form4s(cik):
    """All Form 4 (accession, filingDate, primaryDoc) incl. paginated history."""
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=H, timeout=30).json()
    time.sleep(0.11)
    out, rec = [], sub["filings"]["recent"]
    out += [(rec["accessionNumber"][i], rec["filingDate"][i], rec["primaryDocument"][i])
            for i in range(len(rec["form"])) if rec["form"][i] == "4"]
    for f in sub["filings"].get("files", []):
        older = requests.get(f"https://data.sec.gov/submissions/{f['name']}", headers=H, timeout=30).json()
        time.sleep(0.11)
        out += [(older["accessionNumber"][i], older["filingDate"][i], older["primaryDocument"][i])
                for i in range(len(older["form"])) if older["form"][i] == "4"]
    return out

if "parsed" not in dir(): parsed = {}          # ticker -> list of buy dicts (checkpoint)
signals = []
todo = [t for t in ep.ticker.unique() if t not in parsed]
print(f"{len(todo)} tickers to parse (checkpoint holds {len(parsed)})")
for n, t in enumerate(todo):
    cik = cikmap.get(t)
    if not cik: parsed[t] = []; continue
    try:
        f4s = all_form4s(cik)
    except Exception: parsed[t] = []; continue
    wins = ep[ep.ticker == t]
    # only parse filings that fall inside (episode start-90d, episode end)
    keep = []
    for acc, fd, pdoc in f4s:
        fdt = pd.Timestamp(fd)
        for _, w in wins.iterrows():
            if w.ep_start - pd.Timedelta(days=90) <= fdt <= w.ep_end:
                keep.append((acc, fd, pdoc)); break
    buys = []
    for acc, fd, pdoc in keep[:120]:
        url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc.replace('-','')}/{pdoc.split('/')[-1]}"
        try:
            x = requests.get(url, headers=H, timeout=30).text
            time.sleep(0.11)
            root = ET.fromstring(re.sub(r"<\?xml[^>]*\?>", "", x, count=1))
        except Exception: continue
        owner = txt(root, ".//reportingOwner/reportingOwnerId/rptOwnerName") or "?"
        plan = txt(root, ".//aff10b5One") in ("1", "true")
        for tr in root.findall(".//nonDerivativeTransaction"):
            c_ = txt(tr, "transactionCoding/transactionCode")
            sh = txt(tr, "transactionAmounts/transactionShares/value")
            pr = txt(tr, "transactionAmounts/transactionPricePerShare/value")
            if c_ == "P" and sh and pr and not plan:
                buys.append({"filed": fd, "owner": owner, "v": float(sh) * float(pr), "px": float(pr)})
    parsed[t] = buys
    if n % 20 == 0: print(f"{n}/{len(todo)}", end=" ")

# cluster signal: within any 90d window inside an episode, >=3 distinct buyers and >=$500K total
for t, buys in parsed.items():
    if not buys: continue
    b = pd.DataFrame(buys); b["filed"] = pd.to_datetime(b.filed)
    b = b.sort_values("filed")
    for i in range(len(b)):
        w = b[(b.filed >= b.filed.iloc[i]) & (b.filed <= b.filed.iloc[i] + pd.Timedelta(days=90))]
        if w.owner.nunique() >= 3 and w.v.sum() >= 500000:
            signals.append({"ticker": t, "signal_date": w.filed.max(), "cluster_$": int(w.v.sum()),
                            "buyers": int(w.owner.nunique()),
                            "ins_avg_px": float((w.v.sum() / (w.v / w.px).sum()))})
            break   # first qualifying cluster per ticker-episode era; dedup below
sig = pd.DataFrame(signals).drop_duplicates(subset=["ticker", "signal_date"])
# dedup: one signal per ticker per 12 months
sig = sig.sort_values(["ticker", "signal_date"])
sig = sig[~((sig.ticker == sig.ticker.shift()) & ((sig.signal_date - sig.signal_date.shift()).dt.days < 365))]
print(f"\n{len(sig)} historical cluster-in-drawdown signals")


In [ ]:
# 3 · GATES + FORWARD RETURNS vs SPY (entry = close the day AFTER signal_date; no lookahead)
rows = []
for _, s in sig.iterrows():
    t = s.ticker
    if t not in px.columns: continue
    ser = px[t].dropna()
    after = ser[ser.index > s.signal_date]
    if len(after) < 130: continue
    entry_dt, entry = after.index[0], float(after.iloc[0])
    # gate: entry within +/-25% of insiders' avg price
    if abs(entry / s.ins_avg_px - 1) > 0.25: continue
    spy_after = spy[spy.index >= entry_dt]
    def fwd(series, d):
        return float(series.iloc[min(d, len(series)-1)] / series.iloc[0] - 1) * 100
    r6, r12 = fwd(after, 126), fwd(after, 252)
    s6, s12 = fwd(spy_after, 126), fwd(spy_after, 252)
    rows.append({"ticker": t, "entry": entry_dt.date(), "cluster_$": s["cluster_$"], "buyers": s.buyers,
                 "fwd6m": round(r6,1), "fwd12m": round(r12,1),
                 "alpha6m": round(r6-s6,1), "alpha12m": round(r12-s12,1)})
bt = pd.DataFrame(rows)
print(f"{len(bt)} backtest entries\n")
print(bt.sort_values("entry").to_string(index=False))


In [ ]:
# 4 · THE VERDICT — stats vs the pre-registered hypotheses
print(f"n = {len(bt)} signals, {bt.ticker.nunique()} tickers, {bt.entry.min()} -> {bt.entry.max()}\n")
for col in ["fwd6m", "fwd12m", "alpha6m", "alpha12m"]:
    d = bt[col].dropna()
    print(f"{col:>9}: mean {d.mean():+.1f}% | median {d.median():+.1f}% | win rate {(d>0).mean()*100:.0f}% "
          f"| best {d.max():+.0f}% | worst {d.min():+.0f}%")
print("\nby cluster size:")
for lbl, m in [("$0.5-2M", bt["cluster_$"].between(5e5, 2e6)), ("$2M+", bt["cluster_$"] > 2e6)]:
    d = bt[m]
    if len(d): print(f"  {lbl}: n={len(d)} | median alpha12m {d.alpha12m.median():+.1f}% | win {(d.alpha12m>0).mean()*100:.0f}%")
print("\nby buyer count:")
for lbl, m in [("3-4 buyers", bt.buyers <= 4), ("5+ buyers", bt.buyers >= 5)]:
    d = bt[m]
    if len(d): print(f"  {lbl}: n={len(d)} | median alpha12m {d.alpha12m.median():+.1f}% | win {(d.alpha12m>0).mean()*100:.0f}%")
print("\ntail check (the FISV problem): % of signals with fwd12m < -25%:",
      f"{(bt.fwd12m < -25).mean()*100:.0f}%")


*Grading: paste Cell 4 back. If H1 fails, the live strategy leans entirely on the judgment layer and the vault should say so. If H1 holds, the funnel has a measured base rate under the E-path work.*